# Phase 4a — Hypothesis Tests + Classifier (n = 75, initial sample)Takes the 75 usable records from Phase 3 (`phase3_attention_v2.npz` + metadata, corrected labels) and:- Computes interpretable features from the raw attention tensors, one set per hypothesis- Tests each hypothesis statistically — do corrections differ from hallucination_pivots?- Trains a logistic regression classifier using all features combined- Ablates by category — does the signal hold across math/logic/misdirection?This is the **initial, small-sample** analysis. The encouraging-looking numbers producedhere (ROC-AUC ≈ 0.72) are exactly what Phase 4c goes on to test under stricter validation.**H3 uses the corrected, normalized entropy definition** — see the H3 cell below for why.

In [ ]:
import numpy as npimport jsonimport osfrom collections import Counterimport matplotlib.pyplot as pltimport seaborn as snsfrom statsmodels.stats.multitest import multipletestsfrom sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import StratifiedKFoldfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_scorenp.random.seed(42)NPZ_PATH  = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_attention_v2.npz'META_PATH = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_metadata_v2.jsonl'OUTPUT_DIR = '/kaggle/working/pivot_research'os.makedirs(OUTPUT_DIR, exist_ok=True)N_LAYERS, N_HEADS, N_QUERIES = 28, 12, 30EARLY_KEY_RANGE = 50EARLY_LAYERS = list(range(0, N_LAYERS // 3))            # 0-8LATE_LAYERS  = list(range(2 * N_LAYERS // 3, N_LAYERS))  # 18-27

## Load dataLoads the attention tensors and metadata, and immediately asserts the sample size andlabel counts are what this notebook expects — if `NPZ_PATH` ever points at the wrongfile, this fails here with a clear message instead of silently producing wrong numbersfive cells later.

In [ ]:
npz = np.load(NPZ_PATH)with open(META_PATH) as f:    metadata = [json.loads(l) for l in f if l.strip()]n_records = len(metadata)weights = np.stack([npz[f'weights_{i:03d}'] for i in range(n_records)], axis=0)outputs = np.stack([npz[f'outputs_{i:03d}'] for i in range(n_records)], axis=0)labels = np.array([m['label'] for m in metadata])categories = np.array([m['category'] for m in metadata])y = (labels == 'correction').astype(int)valid_key_starts = np.array([m['valid_key_start'] for m in metadata])corr_mask, halluc_mask = y == 1, y == 0print(f"n={n_records}: correction={corr_mask.sum()}, hallucination_pivot={halluc_mask.sum()}")assert n_records == 75 and corr_mask.sum() == 14, \    f"Expected n=75, 14 correction -- got n={n_records}, {corr_mask.sum()} correction. Check NPZ_PATH."print("Category distribution:", dict(Counter(categories)))

## Shared utilities`permutation_test` is the core statistic used by every hypothesis below: a two-sidedpermutation test on the difference of class means. `run_per_head_test` applies it toall 336 (layer, head) pairs at once and returns Benjamini–Hochberg FDR-correctedresults — used identically by H1, H2, and H3.

In [ ]:
def permutation_test(group1, group2, n_perm=2000, seed=42):    """Two-sided permutation test on difference of means."""    rng = np.random.default_rng(seed)    observed = group1.mean() - group2.mean()    combined = np.concatenate([group1, group2])    n1 = len(group1)    diffs = np.zeros(n_perm)    for i in range(n_perm):        rng.shuffle(combined)        diffs[i] = combined[:n1].mean() - combined[n1:].mean()    return observed, (np.abs(diffs) >= np.abs(observed)).mean()def run_per_head_test(features):    """Runs the permutation test for every (layer, head), returns FDR-corrected results."""    p_values = np.zeros((N_LAYERS, N_HEADS))    effect_sizes = np.zeros((N_LAYERS, N_HEADS))    for L in range(N_LAYERS):        for H in range(N_HEADS):            diff, p = permutation_test(features[corr_mask, L, H], features[halluc_mask, L, H])            p_values[L, H] = p            effect_sizes[L, H] = diff    reject, adj_p, _, _ = multipletests(p_values.flatten(), alpha=0.05, method='fdr_bh')    adj_p = np.asarray(adj_p).reshape(N_LAYERS, N_HEADS)    reject = np.asarray(reject).reshape(N_LAYERS, N_HEADS)    return p_values, effect_sizes, adj_p, reject

## H1 — Early-Context Attention Spike**Claim:** genuine corrections involve the model attending more strongly back toward theearly portion of the captured context window than hallucination pivots do.**Feature:** for each head, total attention mass on the first 50 real (non-padding) keypositions, averaged over the 30 pre-pivot query positions.**Statistical test:** permutation test per head, FDR-corrected across all 336 heads.

In [ ]:
def compute_h1(weights, valid_key_starts):    h1 = np.zeros((n_records, N_LAYERS, N_HEADS))    for i in range(n_records):        v = valid_key_starts[i]        early_end = min(v + EARLY_KEY_RANGE, weights.shape[-1])        h1[i] = weights[i, :, :, :, v:early_end].sum(axis=-1).mean(axis=-1)    return h1h1_features = compute_h1(weights, valid_key_starts)p_values, effect_sizes, adj_p, reject = run_per_head_test(h1_features)print(f"H1: {reject.sum()}/336 significant (FDR); best raw p = {p_values.min():.4f}")

## H2 — Consistency-Head Activation**Claim:** specific heads fire with higher magnitude before genuine corrections thanbefore hallucination pivots (by analogy to functionally specialized heads foundelsewhere in the literature, e.g. induction heads).**Feature:** L2 norm of each head's output vector, averaged over the 30 pre-pivot querypositions.**Statistical test:** same as H1.

In [ ]:
h2_features = np.linalg.norm(outputs, axis=-1).mean(axis=2)   # (n_records, N_LAYERS, N_HEADS)p_values_h2, effect_sizes_h2, adj_p_h2, reject_h2 = run_per_head_test(h2_features)print(f"H2: {reject_h2.sum()}/336 significant (FDR); best adj p = {adj_p_h2.min():.4f}")

## H3 — Attention Entropy (normalized)**Claim:** genuine corrections are preceded by more diffuse (higher-entropy) attention;hallucination pivots by more focused (lower-entropy) attention.**Feature:** Shannon entropy of the attention distribution over real key positions,**normalized by log(K_q)** — K_q is the number of real, non-negligible-weight keypositions available to that specific query position.This normalization matters: raw entropy's ceiling is log(K), so two attention patternsthat are equally diffuse *relative to their own available key-space* can showsystematically different raw entropy purely because K differs by record (padding,truncation) and by query position (causal masking). Dividing by log(K_q) removes thisconfound. An earlier version of this analysis used raw (unnormalized) entropy; see thepaper's Limitations for what changed when this was corrected.The assertion below checks the corrected formula is actually being used — raw entropywould give a p<0.05 rate near 1%, the corrected one gives ~10.7% at n=75.

In [ ]:
def entropy_normalized(row, valid_start):    """Raw Shannon entropy / log(K_q), K_q = real non-negligible keys for this query."""    real = row[valid_start:]    real = real[real > 1e-10]    K_q = len(real)    if K_q <= 1 or real.sum() == 0:        return 0.0    p = real / real.sum()    raw = -np.sum(p * np.log(p))    return raw / np.log(K_q)def compute_h3(weights, valid_key_starts):    h3 = np.zeros((n_records, N_LAYERS, N_HEADS))    for i in range(n_records):        v = valid_key_starts[i]        for L in range(N_LAYERS):            for H in range(N_HEADS):                ents = [entropy_normalized(weights[i, L, H, q, :], v) for q in range(N_QUERIES)]                h3[i, L, H] = np.mean(ents)    return h3h3_features = compute_h3(weights, valid_key_starts)p_values_h3, effect_sizes_h3, adj_p_h3, reject_h3 = run_per_head_test(h3_features)pct_below_05 = (p_values_h3 < 0.05).mean()print(f"H3: {reject_h3.sum()}/336 significant (FDR); best adj p = {adj_p_h3.min():.4f}; "      f"{pct_below_05:.1%} raw p<0.05 (expect ~10.7% at n=75)")assert 0.08 < pct_below_05 < 0.14, \    f"H3 rate {pct_below_05:.1%} outside expected ~10.7% range -- check entropy_normalized is being used"

## H4 — Late/Early Layer Magnitude Ratio**Claim:** hallucination pivots show more late-layer dominance (associated with surfacefluency) relative to early-layer activity (associated with semantic processing) thangenuine corrections do — a motivating heuristic, not an established property of thismodel.**Feature:** ratio of mean H2 magnitude across late layers (18–27) to early layers(0–8), a single scalar per record.**Statistical test:** one permutation test (not per-head, since this is a singlescalar).

In [ ]:
early_mag = h2_features[:, EARLY_LAYERS, :].mean(axis=(1, 2))late_mag  = h2_features[:, LATE_LAYERS, :].mean(axis=(1, 2))h4_features = late_mag / (early_mag + 1e-10)diff_h4, p_h4 = permutation_test(h4_features[halluc_mask], h4_features[corr_mask], n_perm=10000)print(f"H4: diff={diff_h4:+.4f}, p={p_h4:.4f}")

## Figure 3 — Hypothesis heatmap2×2 grid: H1–H3 as per-head FDR-adjusted significance heatmaps, H4 as a distributionhistogram. This is the figure referenced as Figure 3 in the paper.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))ax = axes[0, 0]sns.heatmap(-np.log10(adj_p + 1e-10), ax=ax, cmap='RdYlGn_r', vmin=0, vmax=3, cbar_kws={'label': '-log10(adj p)'})ax.set_title(f'H1: Early-Context Attention Spike\n({reject.sum()} significant heads)')ax.set_xlabel('Head'); ax.set_ylabel('Layer')ax = axes[0, 1]sns.heatmap(-np.log10(adj_p_h2 + 1e-10), ax=ax, cmap='RdYlGn_r', vmin=0, vmax=3, cbar_kws={'label': '-log10(adj p)'})ax.set_title(f'H2: Consistency Head Activation\n({reject_h2.sum()} significant heads)')ax.set_xlabel('Head'); ax.set_ylabel('Layer')ax = axes[1, 0]sns.heatmap(-np.log10(adj_p_h3 + 1e-10), ax=ax, cmap='RdYlGn_r', vmin=0, vmax=3, cbar_kws={'label': '-log10(adj p)'})ax.set_title(f'H3: Attention Entropy (normalized)\n({reject_h3.sum()} significant heads)')ax.set_xlabel('Head'); ax.set_ylabel('Layer')ax = axes[1, 1]ax.hist(h4_features[corr_mask], bins=10, alpha=0.6, label='correction', color='green')ax.hist(h4_features[halluc_mask], bins=20, alpha=0.6, label='hallucination_pivot', color='red')ax.axvline(h4_features[corr_mask].mean(), color='darkgreen', linestyle='--', label='correction mean')ax.axvline(h4_features[halluc_mask].mean(), color='darkred', linestyle='--', label='halluc mean')ax.set_title(f'H4: Late/Early Layer Ratio\np = {p_h4:.4f}')ax.set_xlabel('Late-layer / early-layer magnitude ratio'); ax.set_ylabel('count')ax.legend()plt.tight_layout()plt.savefig(f'{OUTPUT_DIR}/fig3_hypothesis_heatmaps_n75.png', dpi=120, bbox_inches='tight')plt.show()print(f"Saved Figure 3 to {OUTPUT_DIR}/fig3_hypothesis_heatmaps_n75.png")

## Combined classifierConcatenates all four feature sets into one 1,009-dimensional vector (336 + 336 + 336 + 1)and trains an L2-regularized logistic regression classifier, evaluated via stratified5-fold cross-validation. `C=0.01` (heavy regularization) was fixed before any of theseanalyses were run and held constant throughout — see the paper's Methodology for why.This is the **instance-level** baseline: different samples from the same prompt can landin different folds here. Phase 4c re-runs this under prompt-grouping and a fully disjointevaluation to test whether that matters.

In [ ]:
X = np.concatenate([    h1_features.reshape(n_records, -1),    h2_features.reshape(n_records, -1),    h3_features.reshape(n_records, -1),    h4_features.reshape(n_records, 1),], axis=1)print(f"Feature matrix: {X.shape}")skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)all_preds, all_probas = np.zeros(n_records), np.zeros(n_records)fold_results = []for fold_idx, (tr, te) in enumerate(skf.split(X, y)):    sc = StandardScaler()    Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)    clf.fit(Xtr, y[tr])    preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]    all_preds[te], all_probas[te] = preds, probas    ba = balanced_accuracy_score(y[te], preds)    fold_results.append(ba)    print(f"  Fold {fold_idx}: bal_acc={ba:.3f}  (n_correction={int((y[te]==1).sum())})")print(f"\nOverall: bal_acc={balanced_accuracy_score(y, all_preds):.3f}, "      f"F1={f1_score(y, all_preds, zero_division=0):.3f}, "      f"ROC-AUC={roc_auc_score(y, all_probas):.3f}")print(f"Fold mean±SD: {np.mean(fold_results):.3f} ± {np.std(fold_results):.3f}")

## Cross-domain ablationTrains on two of the three prompt categories, evaluates on the third — tests whether anysignal generalizes across domains or is domain-specific. Uses the same classifierconfiguration as above.

In [ ]:
print("Cross-domain ablation (train on 2 categories, test on 3rd):")for held_out in ['math', 'logic', 'misdirection']:    test_mask = categories == held_out    train_mask = ~test_mask    sc = StandardScaler()    Xtr, Xte = sc.fit_transform(X[train_mask]), sc.transform(X[test_mask])    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)    clf.fit(Xtr, y[train_mask])    preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]    ba = balanced_accuracy_score(y[test_mask], preds)    auc = roc_auc_score(y[test_mask], probas)    print(f"  {held_out}: n={test_mask.sum()}, bal_acc={ba:.3f}, AUC={auc:.3f}")

## Save resultsWrites everything to JSON for the paper writeup.

In [ ]:
results = {    'n_records': n_records,    'h1': {'n_sig': int(reject.sum()), 'best_adj_p': float(adj_p.min())},    'h2': {'n_sig': int(reject_h2.sum()), 'best_adj_p': float(adj_p_h2.min())},    'h3': {'n_sig': int(reject_h3.sum()), 'best_adj_p': float(adj_p_h3.min())},    'h4': {'p_value': float(p_h4), 'diff': float(diff_h4)},    'classifier': {        'balanced_accuracy': float(balanced_accuracy_score(y, all_preds)),        'f1': float(f1_score(y, all_preds, zero_division=0)),        'roc_auc': float(roc_auc_score(y, all_probas)),    },}with open(f'{OUTPUT_DIR}/phase4a_n75_results.json', 'w') as f:    json.dump(results, f, indent=2)print(f"Saved to {OUTPUT_DIR}/phase4a_n75_results.json")